# ДЗ 07 · CNN-07 / Homework 07 · CNN-07

**100 баллов / points.** Пять функций, отдельная работа от практики. / Five functions, separate from practice.

RU: Заполните имя и группу. Реализуйте функции, выполните Restart & Run All, добавьте обсуждение и сдайте один .ipynb с выводами. Проверочные ячейки не менять.

EN: Enter your name and group, implement the functions, Restart & Run All, add the discussion and submit one executed .ipynb. Preserve grader cells.

[Сроки по группам / Group deadlines](https://docs.google.com/spreadsheets/d/15N5GGHujXPAzRNuyOfmooT7A7sKhXgrMfWHkUkvYYiM/edit). Срок ДЗ 07 назначает преподаватель; пустая дата не означает автоматическую просрочку. / The instructor sets HW07 dates; blank dates do not imply lateness.

Итог / Final = raw × max(0,1−late_seconds/(14×86400)) × practice_factor. practice_factor=1 при отметке лекции 07, иначе 0.9; проверяющий читает живую таблицу и время Drive, ноутбук выдаёт только raw. / The external reviewer uses live attendance and trusted Drive times; this notebook reports raw points only. The best approved final attempt is retained.

**Данные / Data:** UCI Optical Recognition of Handwritten Digits, E. Alpaydin & C. Kaynak (1998), [DOI:10.24432/C50P49](https://doi.org/10.24432/C50P49), [UCI](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), [sklearn loader](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). UCI: CC BY 4.0, сохраняйте авторство / retain attribution.

RU: `load_digits` содержит 1797 реальных изображений 8×8 из исходной тестовой части UCI. Наблюдение — рукописная цифра; признаки — 64 агрегированных значения интенсивности 0…16; цель — класс 0…9. Это не фотографии высокого разрешения и не ImageNet. Делим на 16 по известному диапазону. Учебное стратифицированное разбиение seed=707: 1078 train, 359 validation, 360 test. Идентификаторов авторов нет, независимость почерков не гарантируется. Это не официальный benchmark. В предыдущей лекции использовались те же исходные данные; новая разбивка не превращает их в неизвестный внешний тест. Сравнения служат учебной демонстрацией. Test не используем для настройки.

EN: `load_digits` contains 1797 real 8×8 images copied from the original UCI test portion. One observation is a handwritten digit; features are 64 aggregated intensity values in 0…16; the target is a class in 0…9. These are not high-resolution photographs or ImageNet. Divide by the known range 16. Teaching stratified split, seed=707: 1078 train, 359 validation, 360 test. Writer IDs are unavailable; writer independence is not guaranteed. This is not the official benchmark. The previous lecture used the same underlying data; a new split does not make these an unseen external test. Comparisons are teaching demonstrations. Never tune using test.

**Среда / Environment:** Google Colab, CPU; NumPy, scikit-learn, matplotlib, PyTorch. Dataset bundled with sklearn; no synthetic fallback. / Данные входят в sklearn, синтетической замены нет. Run cells top to bottom after Restart / Запускайте сверху вниз после Restart.

In [ ]:
student_first = ""
student_last = ""
student_group = ""
assignment_id = "CNN-07"

In [ ]:
import copy  # RU: Копирование весов. EN: Copy model weights.
import json  # RU: Сохранение результатов. EN: Save results.
import numpy as np  # RU: Массивы и ручные операции. EN: Arrays and manual operations.
import matplotlib.pyplot as plt  # RU: Строим графики. EN: Draw plots.
import torch  # RU: Тензоры и обучение. EN: Tensors and training.
from sklearn.datasets import load_digits  # RU: Реальные цифры UCI. EN: Real UCI digits.
from sklearn.model_selection import train_test_split  # RU: Стратификация разбиения. EN: Stratified splitting.
from sklearn.metrics import ConfusionMatrixDisplay  # RU: Матрица ошибок. EN: Confusion matrix.
torch.set_num_threads(1)  # RU: Быстрый CPU для малых данных. EN: One CPU thread suits this small dataset.
torch.manual_seed(707)  # RU: Повторяемая инициализация. EN: Reproducible initialization.
digits = load_digits()  # RU: Загружаем 1797 наблюдений. EN: Load 1797 observations.
images = digits.images.astype(np.float32) / 16.0  # RU: Известный диапазон, без подгонки. EN: Known range, no fitted scaling.
labels = digits.target  # RU: Метки 0–9. EN: Labels 0–9.
indices = np.arange(len(labels))  # RU: Индексы исходных наблюдений. EN: Original observation indices.
train_ids, rest_ids = train_test_split(indices, test_size=0.4, stratify=labels, random_state=707)  # RU: Отделяем train. EN: Separate training data.
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, stratify=labels[rest_ids], random_state=707)  # RU: Разделяем validation и test. EN: Split validation and test.
x = torch.tensor(images[:, None], dtype=torch.float32)  # RU: Добавляем канал: N×1×8×8. EN: Add channel: N×1×8×8.
y = torch.tensor(labels, dtype=torch.long)  # RU: Целые метки для cross-entropy. EN: Integer labels for cross-entropy.
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))  # RU: Проверяем непересечение. EN: Verify disjoint splits.

## correlate2d · 25/100

RU: Реализуйте одноканальную кросс-корреляцию NumPy без переворота ядра. Входы: конечные 2D float-массивы, положительный целый stride, неотрицательный целый padding; ядро помещается в дополненный вход. Нулевое дополнение, dilation=1, floor-размер. Верните 2D float-массив, не меняйте входы. Не вызывайте torch/OpenCV/SciPy внутри решения. Проверьте на crop реальной цифры 7×6 с ядром 3×3 и двух комбинациях stride/padding.
EN: Implement single-channel NumPy cross-correlation without flipping the kernel. Inputs are finite 2D floating arrays, a positive integer stride and nonnegative integer padding; the kernel fits the padded input. Use zero padding, dilation=1 and floor output sizing. Return a 2D floating array; preserve inputs. Do not call torch/OpenCV/SciPy inside the implementation. Verify a real 7×6 digit crop with a 3×3 kernel and two stride/padding combinations.

In [ ]:
def correlate2d(image, kernel, stride=1, padding=0):
    raise NotImplementedError("Implement correlate2d")

## pool2d · 20/100

RU: Реализуйте max pooling 2D NumPy без padding. size и stride — положительные целые; полное окно помещается. Пропускайте неполные окна справа/снизу, сохраните dtype и вход. Размер не обязан делиться на stride. Сравните на реальной карте с F.max_pool2d, включая size=3, stride=2. Не используйте torch внутри функции.
EN: Implement 2D NumPy max pooling without padding. size and stride are positive integers and a full window fits. Skip incomplete right/bottom windows, preserve dtype and input. Input size need not be divisible by stride. Compare a real feature map with F.max_pool2d, including size=3, stride=2. Do not use torch inside this function.

In [ ]:
def pool2d(image, size=2, stride=2):
    raise NotImplementedError("Implement pool2d")

## build_cnn · 20/100

RU: Верните nn.Sequential: Conv2d(1,8,3,padding=1,bias=True), ReLU, MaxPool2d(2), Conv2d(8,16,3,padding=1,bias=True), ReLU, MaxPool2d(2), Flatten, Linear(64,10,bias=True). Выход — logits, без softmax. Общее число параметров 1898. Покажите формы для батча из 5 реальных train-цифр, объясните формулу количества параметров.
EN: Return the exact nn.Sequential architecture: Conv2d(1,8,3,padding=1,bias=True), ReLU, MaxPool2d(2), Conv2d(8,16,3,padding=1,bias=True), ReLU, MaxPool2d(2), Flatten, Linear(64,10,bias=True). Output logits without softmax. Total parameter count: 1898. Show shapes for five real training digits and explain the parameter-count formula.

In [ ]:
def build_cnn():
    raise NotImplementedError("Implement build_cnn")

## evaluate · 20/100

RU: Для непустого CPU-батча верните dict с Python float loss и accuracy. Loss — средняя cross-entropy по logits, accuracy — доля верных argmax. Временно включите eval, выполняйте forward внутри no_grad, восстановите исходный режим model.training. Не меняйте параметры и существующие градиенты. Допустима модель с Dropout: train и eval не взаимозаменяемы. Продемонстрируйте проверку режима и неизменности весов.
EN: For a nonempty CPU batch, return a dict with Python float loss and accuracy. Loss is mean cross-entropy from logits; accuracy is the fraction of correct argmax predictions. Temporarily enable eval, run forward inside no_grad and restore the original model.training mode. Preserve parameters and existing gradients. The model may contain Dropout: train and eval are not interchangeable. Demonstrate mode restoration and unchanged weights.

In [ ]:
def evaluate(model, inputs, targets):
    raise NotImplementedError("Implement evaluate")

## select_best · 15/100

RU: records — непустой список словарей с конечным val_loss. Верните Python int, индекс минимального val_loss, при равенстве первый. Не выбирайте по train_loss, accuracy или test. Запустите эксперимент ниже: две заранее заданные скорости, checkpoint по validation, один финальный test. Объясните выбранную эпоху и ограничения разбиения.
EN: records is a nonempty list of dictionaries with finite val_loss. Return a Python int index of the smallest val_loss, taking the first tie. Do not select by train_loss, accuracy or test. Run the experiment below: two predeclared learning rates, validation checkpointing and one final test. Explain the selected epoch and split limitations.

In [ ]:
def select_best(records):
    raise NotImplementedError("Implement select_best")

## Эксперимент / Experiment

RU: Настройки фиксированы до test: seed707, Adam, batch64, 25 эпох, lr0.001/0.003. Выберите по validation. Сохраните график, лучший epoch, val_loss и test accuracy. Автоматические баллы — за функции; вывод обсуждается с преподавателем, скрытых баллов за стиль нет.

EN: Settings fixed before test: seed707, Adam, batch64, 25 epochs, lr0.001/0.003. Select by validation. Retain the curve, best epoch, val_loss and test accuracy. Automated points assess functions; the instructor discusses conclusions without hidden style points.

In [ ]:
def fit(lr, epochs=25):  # RU: Обучение одного заранее заданного кандидата. EN: Train one predeclared candidate.
    torch.manual_seed(707)  # RU: Одинаковая начальная точка кандидатов. EN: Identical candidate initialization.
    model = build_cnn()  # RU: Создаём новую модель. EN: Create a fresh model.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # RU: Выбираем Adam. EN: Use Adam optimization.
    rng = np.random.default_rng(707)  # RU: Повторяемое перемешивание реальных объектов. EN: Reproducible shuffling of real observations.
    best_loss = float('inf')  # RU: До первой эпохи лучшего loss нет. EN: No best loss before the first epoch.
    history = []  # RU: Накапливаем метрики эпох. EN: Collect epoch metrics.
    for epoch in range(epochs):  # RU: Выполняем заданное число эпох. EN: Run the fixed number of epochs.
        model.train()  # RU: Включаем обучение. EN: Enable training mode.
        order = rng.permutation(train_ids)  # RU: Перемешиваем только train. EN: Shuffle training data only.
        for start in range(0, len(order), 64):  # RU: Мини-батчи по 64. EN: Mini-batches of 64.
            batch = order[start:start + 64]  # RU: Индексы текущего батча. EN: Current batch indices.
            optimizer.zero_grad(set_to_none=True)  # RU: Сбрасываем старые градиенты. EN: Clear old gradients.
            loss = torch.nn.functional.cross_entropy(model(x[batch]), y[batch])  # RU: Loss по logits, без softmax заранее. EN: Loss from logits, without prior softmax.
            loss.backward()  # RU: Рассчитываем градиенты. EN: Compute gradients.
            optimizer.step()  # RU: Обновляем параметры. EN: Update parameters.
        train_result = evaluate(model, x[train_ids], y[train_ids])  # RU: Метрика train после эпохи. EN: Training metric after the epoch.
        val_result = evaluate(model, x[val_ids], y[val_ids])  # RU: Только validation выбирает checkpoint. EN: Only validation selects the checkpoint.
        history.append([train_result['loss'], val_result['loss']])  # RU: Сохраняем кривые. EN: Save learning curves.
        if val_result['loss'] < best_loss:  # RU: Обновляем только при строгом улучшении. EN: Update only on strict improvement.
            best_loss = val_result['loss']  # RU: Запоминаем лучший loss. EN: Remember the best loss.
            best_state = copy.deepcopy(model.state_dict())  # RU: Копируем, а не сохраняем ссылку. EN: Copy weights rather than retaining a reference.
            best_epoch = epoch + 1  # RU: Эпохи нумеруем с единицы. EN: Epoch numbering starts at one.
    model.load_state_dict(best_state)  # RU: Восстанавливаем лучший checkpoint. EN: Restore the best checkpoint.
    model.eval()  # RU: Подготавливаем модель к оценке. EN: Prepare the model for evaluation.
    return model, {'lr': lr, 'val_loss': best_loss, 'best_epoch': best_epoch, 'history': history}  # RU: Возвращаем модель и журнал. EN: Return the model and log.
experiment_result = None
try:
    candidates = [fit(lr) for lr in [0.001, 0.003]]
    reports = [report for _, report in candidates]
    selected = select_best(reports)
    chosen, report = candidates[selected]
    test_result = evaluate(chosen, x[test_ids], y[test_ids])
    experiment_result = {'selected': int(selected), 'lr': report['lr'], 'epoch': report['best_epoch'], 'val_loss': report['val_loss'], 'test': test_result}
    print(json.dumps(experiment_result, indent=2))
    for record in reports:
        plt.plot(np.arange(1,26), np.array(record['history'])[:,1], label=f"lr={record['lr']}")
    plt.xlabel('epoch'); plt.ylabel('validation cross-entropy'); plt.legend(); plt.show()
except NotImplementedError:
    print('Реализуйте функции и перезапустите / Implement functions and rerun')

## Подробный вывод / Detailed discussion

RU: 1. Укажите выбранные lr и epoch, сравните оба validation loss. 2. Покажите до трёх реальных ошибок test с истинной и предсказанной меткой; если их меньше, сообщите сколько. 3. Для каждого изображения опишите визуальную гипотезу ошибки. 4. Объясните, почему число параметров не доказывает превосходство CNN. 5. Назовите ограничения по авторам и разрешению. 6. Объясните разницу eval и no_grad. Не перенастраивайте сеть после просмотра test.

EN: 1. State the selected lr and epoch and compare both validation losses. 2. Show up to three actual test mistakes with true/predicted labels; report if fewer exist. 3. Give a visual hypothesis for each. 4. Explain why parameter count does not prove CNN superiority. 5. State writer and resolution limitations. 6. Explain eval versus no_grad. Do not retune after viewing test.

**Ваш ответ / Your answer:**

In [ ]:
# Изображения ошибок и пояснения / Error images and explanation

In [ ]:
# Trusted checks: preserve this cell / Доверенные проверки: не изменять.
checks = []
def check(name, points, fn):
    try:
        fn(); checks.append({'test':name, 'points':points, 'passed':True})
    except Exception as exc:
        checks.append({'test':name, 'points':0, 'passed':False, 'error':type(exc).__name__+': '+str(exc)[:180]})
def test_correlation():
    kernels = [np.array([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=float), images[train_ids[1]][:2,:3].astype(float)]
    for index in train_ids[:3]:
        for kernel in kernels:
            for stride,padding in [(1,0),(1,1),(2,1)]:
                image = images[index][:7,:6].astype(float); old=image.copy(); oldk=kernel.copy()
                expected = torch.nn.functional.conv2d(torch.tensor(image[None,None]),torch.tensor(kernel[None,None]),stride=stride,padding=padding)[0,0].numpy()
                got=correlate2d(image,kernel,stride,padding)
                assert isinstance(got,np.ndarray) and got.shape==expected.shape and np.issubdtype(got.dtype,np.floating)
                np.testing.assert_allclose(got,expected,atol=1e-10)
                np.testing.assert_array_equal(image,old);np.testing.assert_array_equal(kernel,oldk)
def test_pool():
    for index in train_ids[:4]:
        image=images[index][:7,:6];old=image.copy()
        for size,stride in [(2,2),(3,2),(2,1)]:
            expected=torch.nn.functional.max_pool2d(torch.tensor(image[None,None]),size,stride)[0,0].numpy()
            got=pool2d(image,size,stride)
            assert isinstance(got,np.ndarray) and got.dtype==image.dtype
            np.testing.assert_array_equal(got,expected);np.testing.assert_array_equal(image,old)
def test_architecture():
    model=build_cnn();assert isinstance(model,torch.nn.Sequential)
    expected_types=[torch.nn.Conv2d,torch.nn.ReLU,torch.nn.MaxPool2d,torch.nn.Conv2d,torch.nn.ReLU,torch.nn.MaxPool2d,torch.nn.Flatten,torch.nn.Linear]
    assert len(model)==len(expected_types)
    assert all(isinstance(layer,kind) for layer,kind in zip(model,expected_types))
    assert sum(p.numel() for p in model.parameters())==1898
    for index,channels in [(0,(1,8)),(3,(8,16))]:
        layer=model[index];assert (layer.in_channels,layer.out_channels)==channels
        assert layer.kernel_size==(3,3) and layer.padding==(1,1) and layer.stride==(1,1) and layer.dilation==(1,1) and layer.groups==1 and layer.bias is not None
    for index in [2,5]:
        layer=model[index];assert layer.kernel_size in (2,(2,2)) and layer.stride in (2,(2,2)) and layer.padding in (0,(0,0)) and not layer.ceil_mode
    assert model[6].start_dim==1 and model[6].end_dim==-1
    assert model[7].in_features==64 and model[7].out_features==10 and model[7].bias is not None
    model.train();model.zero_grad(set_to_none=True)
    value=x[train_ids[:5]];expected_shapes=[(5,8,8,8),(5,8,8,8),(5,8,4,4),(5,16,4,4),(5,16,4,4),(5,16,2,2),(5,64),(5,10)]
    for layer,shape in zip(model,expected_shapes):
        value=layer(value);assert tuple(value.shape)==shape
    assert torch.isfinite(value).all()
    torch.nn.functional.cross_entropy(value,y[train_ids[:5]]).backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
def test_evaluation():
    class Probe(torch.nn.Sequential):
        def forward(self,inputs):
            self.observed=(self.training,torch.is_grad_enabled())
            return super().forward(inputs)
    torch.manual_seed(17)
    model=Probe(torch.nn.Flatten(),torch.nn.Dropout(.5),torch.nn.Linear(64,10))
    model.eval()
    with torch.no_grad():
        z=model(x[val_ids[:19]]);expected_loss=torch.nn.functional.cross_entropy(z,y[val_ids[:19]]).item();expected_acc=(z.argmax(1)==y[val_ids[:19]]).float().mean().item()
    snapshot=copy.deepcopy(model.state_dict())
    for training in [True,False]:
        model.train(training)
        for p in model.parameters():p.grad=torch.ones_like(p)
        got=evaluate(model,x[val_ids[:19]],y[val_ids[:19]])
        assert set(got)=={'loss','accuracy'} and all(isinstance(v,float) for v in got.values())
        assert model.observed==(False,False) and model.training==training
        assert np.isclose(got['loss'],expected_loss) and np.isclose(got['accuracy'],expected_acc)
        assert all(torch.equal(snapshot[k],v) for k,v in model.state_dict().items())
        assert all(torch.equal(p.grad,torch.ones_like(p)) for p in model.parameters())
def test_selection():
    records=[]
    for seed in [1,2,3]:
        torch.manual_seed(seed);model=torch.nn.Sequential(torch.nn.Flatten(),torch.nn.Linear(64,10))
        with torch.no_grad():vl=torch.nn.functional.cross_entropy(model(x[val_ids[:20]]),y[val_ids[:20]]).item()
        records.append({'val_loss':vl,'train_loss':-vl})
    for values in [records,records[::-1],[records[0],records[0]],records[1:]]:
        before=copy.deepcopy(values);got=select_best(values)
        assert type(got) is int and got==min(range(len(values)),key=lambda i:values[i]['val_loss'])
        assert values==before
check('correlation',25,test_correlation)
check('pooling',20,test_pool)
check('architecture',20,test_architecture)
check('evaluation',20,test_evaluation)
check('selection',15,test_selection)
raw_score=sum(c['points'] for c in checks)
print(json.dumps(checks,ensure_ascii=False,indent=2))
print(json.dumps({'student_first':student_first.strip(),'student_last':student_last.strip(),'student_group':student_group.strip(),'assignment_id':'CNN-07','name':(student_first+' '+student_last).strip(),'group':student_group.strip(),'assignment':'CNN-07','score':raw_score,'raw_score':raw_score,'max_score':100,'score_kind':'raw'},ensure_ascii=False))